In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import spearmanr
import joblib

In [2]:
df = pd.read_csv("datasets/final_ds.csv")
df_clean = df.dropna(subset=['NCAR20', 'stage', 'date']).reset_index(drop=True)

In [3]:
df_clean['date'] = pd.to_datetime(df_clean['date'])

In [4]:
train_mask = (df_clean['date'] >= '2025-01-01') & (df_clean['date'] <= '2025-11-30')
test_mask = (df_clean['date'] >= '2026-01-01') & (df_clean['date'] <= '2026-06-30')

In [ ]:
models = {}
eval_metrics = []

for dayX in range(1,20)
target = f'NCAR{dayX}'
features = ['stage', 
            'polarity',
            'drug_count',
            'Total Revenue', 
            'Operating Revenue', 
            'Cash And Cash Equivalents', 
            'Capital Expenditure', 
            'Stock_Trend_30d', 
            'Stock_Trend_60d',
            'NBI_Trend_30d', 
            'NBI_Trend_60d',
            'isDelisted']


X_train = df_clean.loc[train_mask, features]
y_train = df_clean.loc[train_mask, target]

X_test = df_clean.loc[test_mask, features]
y_test = df_clean.loc[test_mask, target]

X_train['stage'] = X_train['stage'].astype('category')
X_test['stage'] = X_test['stage'].astype('category')

X_train['isDelisted'] = X_train['isDelisted'].astype('int')
X_test['isDelisted'] = X_test['isDelisted'].astype('int')

cont_vars = [c for c in features if (c != 'stage' and c!= 'isDelisted')]
X_train[cont_vars] = X_train[cont_vars].apply(pd.to_numeric)
X_test[cont_vars] = X_test[cont_vars].apply(pd.to_numeric)

y_train = y_train.apply(pd.to_numeric)
y_test = y_test.apply(pd.to_numeric)

print(f"Train observation count: {len(X_train)}")
print(f"Test observation count: {len(X_test)}")

model = xgb.XGBRegressor(
    objective = 'reg:pseudohubererror',
    huber_slope = 1.0,
    eval_metric = 'mae',
    n_estimators = 100,
    learning_rate = 0.05,
    max_depth=5,
    random_state = 42,
    n_jobs = 1,
    tree_method = "hist",
    enable_categorical = True,
    missing = np.nan
)


model.fit(X_train,
          y_train,
          eval_set = [(X_test, y_test)],
          verbose= False
          )

models[target] = model

y_pred = model.predict(X_test)

# model evaluation

rank_ic, p_val = spearmanr(y_test, y_pred)

eval_metrics.append({
    "Day": dayX,
    "Directional Accuracy": np.mean(np.sign(y_pred) == np.sign(y_test)),
    "Rank IC": rank_ic, 
    "p_val": p_val,
    "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
    "MAE": mean_absolute_error(y_test, y_pred),
    "R2": r2_score(y_test, y_pred)
})

In [ ]:
print(eval_metrics)

In [ ]:
importance = pd.DataFrame({
    'Features' : features,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending = False)

print(importance)

In [ ]:
labels = range(1,11)

deciles = pd.qcut(y_pred, q=10, labels = labels)
spread = y_test.groupby(deciles).mean()

print(spread)

In [ ]:
joblib.dump(model, "xgb_model.joblib")

['xgb_model.joblib']